# HW2 — DCGAN on WikiArt (Kaggle)
**Before running:** *Add Input* → search `steubk/wikiart` → add it. In *Settings*: Accelerator = **GPU P100** (or T4), Internet = **On** (needed for `pip install` and Inception weights).
Then **Save Version → Save & Run All** so it runs in the background (~1–1.5 h total).

In [ ]:
!pip install -q pytorch-ignite
!nvidia-smi -L

In [ ]:
%%writefile models.py
"""DCGAN generator and discriminator for 64x64 RGB images (Radford et al., 2016)."""
import torch.nn as nn


class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super().__init__()
        self.net = nn.Sequential(
            # z: (nz,1,1) -> (ngf*8,4,4)
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8), nn.ReLU(True),
            # -> (ngf*4,8,8)
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            # -> (ngf*2,16,16)
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            # -> (ngf,32,32)
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            # -> (nc,64,64)
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, ndf=64, nc=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),  # logits, (B,1,1,1)
        )

    def forward(self, x):
        return self.net(x).view(-1)


def weights_init(m):
    """DCGAN init: N(0, 0.02) for convs, N(1, 0.02) for BN scale."""
    name = m.__class__.__name__
    if "Conv" in name:
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif "BatchNorm" in name:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)


In [ ]:
%%writefile data.py
import numpy as np
import torch
from torch.utils.data import Dataset


class NpyImages(Dataset):
    """Images cached by preprocess.py, returned as float tensors in [-1, 1]."""

    def __init__(self, path, hflip=True):
        self.x = torch.from_numpy(np.load(path)).permute(0, 3, 1, 2).contiguous()  # (N,3,H,W) uint8
        self.hflip = hflip

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        img = self.x[i].float().div(127.5).sub(1.0)
        if self.hflip and torch.rand(()) < 0.5:
            img = img.flip(-1)
        return img


In [ ]:
%%writefile preprocess.py
"""Resize WikiArt images once to 64x64 and cache them as a uint8 .npy array.

WikiArt originals are large JPEGs; decoding them every epoch is very slow, so we
do it once. After this, the raw dataset can be deleted to free disk space.

Usage:
  python preprocess.py --root data/wikiart --out data/wikiart64.npy
  python preprocess.py --root data/wikiart --styles Impressionism --out data/impr64.npy
"""
import argparse
import os
from concurrent.futures import ProcessPoolExecutor

import numpy as np
from PIL import Image
from tqdm import tqdm

Image.MAX_IMAGE_PIXELS = None
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def load(args):
    path, size = args
    try:
        img = Image.open(path).convert("RGB")
        w, h = img.size
        s = min(w, h)  # center crop to square, then resize
        img = img.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
        img = img.resize((size, size), Image.BICUBIC)
        return np.asarray(img, dtype=np.uint8)
    except Exception:
        return None


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--root", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--size", type=int, default=64)
    p.add_argument("--styles", nargs="*", default=None, help="subfolder names to keep (default: all)")
    p.add_argument("--max-images", type=int, default=None)
    p.add_argument("--workers", type=int, default=os.cpu_count())
    a = p.parse_args()

    paths = []
    for d, _, files in os.walk(a.root):
        if a.styles and os.path.basename(d) not in a.styles:
            continue
        paths += [os.path.join(d, f) for f in files if os.path.splitext(f)[1].lower() in EXTS]
    paths.sort()
    if a.max_images:
        rng = np.random.default_rng(0)
        paths = list(rng.choice(paths, size=min(a.max_images, len(paths)), replace=False))
    print(f"Found {len(paths)} images")

    with ProcessPoolExecutor(a.workers) as ex:
        imgs = list(tqdm(ex.map(load, [(p_, a.size) for p_ in paths], chunksize=64), total=len(paths)))
    imgs = np.stack([im for im in imgs if im is not None])
    np.save(a.out, imgs)
    print(f"Saved {imgs.shape} -> {a.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
"""Train a DCGAN on cached WikiArt images.

Usage: python train.py --data data/wikiart64.npy --epochs 50 --out outputs/run1
"""
import argparse
import json
import os
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.utils import save_image

from data import NpyImages
from models import Discriminator, Generator, weights_init


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", required=True)
    p.add_argument("--out", default="outputs/run1")
    p.add_argument("--epochs", type=int, default=50)
    p.add_argument("--batch-size", type=int, default=128)
    p.add_argument("--nz", type=int, default=100)
    p.add_argument("--ngf", type=int, default=64)
    p.add_argument("--ndf", type=int, default=64)
    p.add_argument("--lr-g", type=float, default=2e-4)
    p.add_argument("--lr-d", type=float, default=2e-4)
    p.add_argument("--beta1", type=float, default=0.5)
    p.add_argument("--label-smooth", type=float, default=0.9, help="target for real labels (1.0 = off)")
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--resume", default=None)
    args = p.parse_args()

    torch.manual_seed(args.seed)
    os.makedirs(f"{args.out}/samples", exist_ok=True)
    with open(f"{args.out}/config.json", "w") as f:
        json.dump(vars(args), f, indent=2)
    device = get_device()
    print("device:", device)

    ds = NpyImages(args.data)
    dl = DataLoader(ds, batch_size=args.batch_size, shuffle=True, drop_last=True,
                    num_workers=args.workers, pin_memory=device.type == "cuda",
                    persistent_workers=args.workers > 0)
    print(f"{len(ds)} images, {len(dl)} iters/epoch")

    G = Generator(args.nz, args.ngf).to(device)
    D = Discriminator(args.ndf).to(device)
    G.apply(weights_init)
    D.apply(weights_init)
    optG = torch.optim.Adam(G.parameters(), lr=args.lr_g, betas=(args.beta1, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=args.lr_d, betas=(args.beta1, 0.999))
    bce = nn.BCEWithLogitsLoss()
    fixed_z = torch.randn(64, args.nz, 1, 1, device=device)

    history = {"g_loss": [], "d_loss": [], "d_real": [], "d_fake": []}
    start_epoch = 0
    if args.resume:
        ck = torch.load(args.resume, map_location=device)
        G.load_state_dict(ck["G"]); D.load_state_dict(ck["D"])
        optG.load_state_dict(ck["optG"]); optD.load_state_dict(ck["optD"])
        history, start_epoch, fixed_z = ck["history"], ck["epoch"] + 1, ck["fixed_z"].to(device)

    for epoch in range(start_epoch, args.epochs):
        G.train(); D.train()
        t0 = time.time()
        for i, real in enumerate(dl):
            real = real.to(device, non_blocking=True)
            b = real.size(0)
            z = torch.randn(b, args.nz, 1, 1, device=device)
            fake = G(z)

            # ---- D step: maximize log D(x) + log(1 - D(G(z)))
            optD.zero_grad(set_to_none=True)
            out_real = D(real)
            out_fake = D(fake.detach())
            d_loss = bce(out_real, torch.full_like(out_real, args.label_smooth)) + \
                     bce(out_fake, torch.zeros_like(out_fake))
            d_loss.backward()
            optD.step()

            # ---- G step: non-saturating loss, maximize log D(G(z))
            optG.zero_grad(set_to_none=True)
            out = D(fake)
            g_loss = bce(out, torch.ones_like(out))
            g_loss.backward()
            optG.step()

            history["g_loss"].append(g_loss.item())
            history["d_loss"].append(d_loss.item())
            history["d_real"].append(torch.sigmoid(out_real).mean().item())
            history["d_fake"].append(torch.sigmoid(out_fake).mean().item())
            if i % 100 == 0:
                print(f"[{epoch+1}/{args.epochs}][{i}/{len(dl)}] D {d_loss.item():.3f} G {g_loss.item():.3f} "
                      f"D(x) {history['d_real'][-1]:.3f} D(G(z)) {history['d_fake'][-1]:.3f}")

        G.eval()
        with torch.no_grad():
            save_image(G(fixed_z), f"{args.out}/samples/epoch_{epoch+1:03d}.png", nrow=8, normalize=True, value_range=(-1, 1))
        ck = {"G": G.state_dict(), "D": D.state_dict(), "optG": optG.state_dict(), "optD": optD.state_dict(),
              "history": history, "epoch": epoch, "fixed_z": fixed_z.cpu(), "args": vars(args)}
        torch.save(ck, f"{args.out}/last.pt")
        if (epoch + 1) % 10 == 0:
            torch.save(G.state_dict(), f"{args.out}/G_epoch{epoch+1:03d}.pt")
        plot_losses(history, len(dl), f"{args.out}/loss_curves.png")
        print(f"epoch {epoch+1} done in {time.time()-t0:.0f}s")

    torch.save(G.state_dict(), f"{args.out}/G_final.pt")
    with open(f"{args.out}/history.json", "w") as f:
        json.dump(history, f)


def smooth(x, k=50):
    if len(x) < k:
        return x
    import numpy as np
    return np.convolve(x, np.ones(k) / k, mode="valid")


def plot_losses(h, iters_per_epoch, path):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    for key, lbl in [("d_loss", "Discriminator"), ("g_loss", "Generator")]:
        ax[0].plot(h[key], alpha=0.2)
        ax[0].plot(range(49, 49 + len(smooth(h[key]))) if len(h[key]) >= 50 else range(len(h[key])),
                   smooth(h[key]), label=lbl, color=ax[0].lines[-1].get_color())
    ax[0].set_xlabel("iteration"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("Training losses")
    ax[1].plot(smooth(h["d_real"]), label="D(x)")
    ax[1].plot(smooth(h["d_fake"]), label="D(G(z))")
    ax[1].set_xlabel("iteration"); ax[1].legend(); ax[1].set_title("Discriminator outputs")
    for a in ax:
        sec = a.secondary_xaxis("top", functions=(lambda x: x / iters_per_epoch, lambda e: e * iters_per_epoch))
        sec.set_xlabel("epoch")
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
"""Compute Inception Score and FID with PyTorch-Ignite, following
https://pytorch-ignite.ai/blog/gan-evaluation-with-fid-and-is/

Usage: python evaluate.py --data data/wikiart64.npy --ckpt outputs/run1/G_final.pt --n 10000
"""
import argparse
import json

import torch
import torch.nn.functional as F
from ignite.engine import Engine
from ignite.metrics import FID, InceptionScore
from ignite.metrics.gan.utils import InceptionModel
from torch.utils.data import DataLoader, Subset
from torchvision.utils import save_image

from data import NpyImages
from models import Generator


def interpolate(batch):
    """[-1,1] 64x64 -> [0,1] 299x299 for Inception-v3."""
    batch = (batch.clamp(-1, 1) + 1) / 2
    return F.interpolate(batch, size=(299, 299), mode="bilinear", align_corners=False)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", required=True)
    p.add_argument("--ckpt", required=True)
    p.add_argument("--n", type=int, default=10000, help="number of real/fake images to compare")
    p.add_argument("--batch-size", type=int, default=100)
    p.add_argument("--nz", type=int, default=100)
    p.add_argument("--ngf", type=int, default=64)
    p.add_argument("--out", default=None, help="json file to write scores to")
    p.add_argument("--samples", default=None, help="save a grid of 64 samples here")
    args = p.parse_args()

    # Ignite's FID accumulates in float64, which MPS doesn't support -> use CUDA or CPU.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(0)

    G = Generator(args.nz, args.ngf).to(device)
    G.load_state_dict(torch.load(args.ckpt, map_location=device))
    G.eval()

    ds = NpyImages(args.data, hflip=False)
    idx = torch.randperm(len(ds))[: args.n]
    dl = DataLoader(Subset(ds, idx.tolist()), batch_size=args.batch_size, shuffle=False, drop_last=True)

    # Blog default: FID on Inception softmax outputs (1000-d). Values are much smaller
    # than the "standard" FID from papers, which uses 2048-d pool3 features.
    fid_metric = FID(device=device)
    fid_pool_metric = FID(num_features=2048, device=device,
                          feature_extractor=InceptionModel(return_features=True, device=device))
    is_metric = InceptionScore(device=device, output_transform=lambda x: x[0])

    def evaluation_step(engine, batch):
        with torch.no_grad():
            z = torch.randn(batch.size(0), args.nz, 1, 1, device=device)
            fake = interpolate(G(z))
            real = interpolate(batch.to(device))
            return fake, real

    evaluator = Engine(evaluation_step)
    fid_metric.attach(evaluator, "fid")
    fid_pool_metric.attach(evaluator, "fid_pool3")
    is_metric.attach(evaluator, "is")
    evaluator.run(dl, max_epochs=1)
    m = evaluator.state.metrics
    res = {"fid": float(m["fid"]), "fid_pool3": float(m["fid_pool3"]), "is": float(m["is"]), "n_images": len(dl) * args.batch_size}
    print(json.dumps(res, indent=2))
    if args.out:
        with open(args.out, "w") as f:
            json.dump(res, f, indent=2)
    if args.samples:
        with torch.no_grad():
            save_image(G(torch.randn(64, args.nz, 1, 1, device=device)), args.samples,
                       nrow=8, normalize=True, value_range=(-1, 1))


if __name__ == "__main__":
    main()


## 1. Locate dataset & cache at 64×64

In [ ]:
import os, glob
cands = [d for d in glob.glob('/kaggle/input/**/Impressionism', recursive=True)]
ROOT = os.path.dirname(cands[0]); print('dataset root:', ROOT)
print(sorted(os.listdir(ROOT))[:40])

In [ ]:
!python preprocess.py --root "$ROOT" --out /kaggle/working/wikiart64.npy --workers 4

## 2. Train

In [ ]:
!python train.py --data /kaggle/working/wikiart64.npy --epochs 50 --batch-size 128 --out /kaggle/working/run1 --workers 2

## 3. Evaluate (IS / FID, 10k real vs 10k fake)

In [ ]:
!python evaluate.py --data /kaggle/working/wikiart64.npy --ckpt /kaggle/working/run1/G_final.pt --n 10000 --out /kaggle/working/run1/scores.json --samples /kaggle/working/run1/final_samples.png

In [ ]:
# Score intermediate checkpoints too (nice for the report: FID vs. epoch)
import json, subprocess
rows = []
for ep in [10, 20, 30, 40, 50]:
    ck = f'/kaggle/working/run1/G_epoch{ep:03d}.pt'
    subprocess.run(['python','evaluate.py','--data','/kaggle/working/wikiart64.npy','--ckpt',ck,'--n','5000','--out',f'/kaggle/working/run1/scores_ep{ep}.json'], check=True, capture_output=True)
    r = json.load(open(f'/kaggle/working/run1/scores_ep{ep}.json')); r['epoch'] = ep; rows.append(r); print(r)
json.dump(rows, open('/kaggle/working/run1/scores_by_epoch.json','w'), indent=2)

## 4. Results

In [ ]:
from IPython.display import Image, display
display(Image('/kaggle/working/run1/loss_curves.png'))
display(Image('/kaggle/working/run1/final_samples.png'))
display(Image('/kaggle/working/run1/samples/epoch_001.png', width=300), Image('/kaggle/working/run1/samples/epoch_050.png', width=300))

In [ ]:
!rm /kaggle/working/wikiart64.npy /kaggle/working/run1/last.pt  # keep outputs small for download
!cd /kaggle/working && zip -qr run1.zip run1 && ls -lh run1.zip